# ZOD02 — full-validation precision-ranked cascade

This notebook uses the full frozen 540-row V0 validation set to rank checkpoint-label rules by Beta(1,1)-smoothed precision. It applies every eligible rule (minimum support 10) in descending precision order to still-unassigned examples, then uses UltraBERT V1-all → V0 as fallback. There is no fold-based selection and no model-weight training. Since rule ranking and scoring use the same V0 labels, the resulting score is an in-sample selection score, not an independent validation estimate. The checkpoint roster remains the 22 already packaged ZOD02 candidates.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from src.zod02 import _validation_frame, _load_cached_candidates, _metrics, _apply_fixed_zod01
from src.zod01 import DEFAULT_MINIMUM_SUPPORT, DEFAULT_PRIOR_ALPHA, DEFAULT_PRIOR_BETA
from src.zod01_fixed_cascade import _resolve_fallback
catalog = pd.read_csv("docs/checkpoint_label_catalog.csv", dtype=str, keep_default_na=False).drop_duplicates("experiment_id").set_index("experiment_id")

validation = _validation_frame()
classes = sorted(validation.y_true.unique().tolist())
fallback_id = str(_resolve_fallback("experiments/registry.sqlite")["experiment_id"])
package = json.loads(Path("artifacts/kaggle/zod02-models/zod02_manifest.json").read_text())
package_candidate_ids = {entry["candidate_id"] for entry in package["candidates"].values()}
_, all_top1, metadata, zod01_rules, exclusions, _ = _load_cached_candidates(
    validation=validation,
    inventory_path="artifacts/analysis/zod01_specialist_discovery/checkpoint_inventory.csv",
    ranked_rules_path="artifacts/analysis/zod01_specialist_discovery/ranked_checkpoint_label_rules.csv",
    predictions_dir="artifacts/analysis/zod01_specialist_discovery/predictions",
    fallback_id=fallback_id)
missing = package_candidate_ids - set(all_top1)
assert not missing, f"packaged candidate prediction missing: {missing}"
top1_by_id = {candidate_id: all_top1[candidate_id] for candidate_id in package_candidate_ids}
y = validation.y_true.to_numpy(dtype=str)


In [2]:
ranked = pd.read_csv("artifacts/analysis/zod01_specialist_discovery/ranked_checkpoint_label_rules.csv", dtype=str, keep_default_na=False)
ranked = ranked[ranked.candidate_id.isin(package_candidate_ids)]
ranked = ranked[ranked.eligible.str.lower().isin(["true", "1"]) & ranked.support_eligible.str.lower().isin(["true", "1"])].copy()
ranked["rank"] = pd.to_numeric(ranked["rank"])
ranked = ranked.sort_values("rank", kind="stable").reset_index(drop=True)
# The source ranking was calculated on all 540 V0 examples using Beta(1,1) smoothing.
rules = [{"order": i + 1, "candidate_id": str(row.candidate_id),
          "sequence_id": str(catalog.loc[row.candidate_id, "sequence_id"]),
          "label": str(row.label), "precision": float(row.precision),
          "smoothed_precision": float(row.smoothed_precision),
          "prediction_count": int(row.prediction_count)}
         for i, row in enumerate(ranked.itertuples())]
assigned = np.zeros(len(y), dtype=bool)
prediction = np.empty(len(y), dtype=object)
selected_candidate = np.empty(len(y), dtype=object)
trace = []
for rule in rules:
    cid, label = rule["candidate_id"], rule["label"]
    rows = np.flatnonzero((~assigned) & (top1_by_id[cid] == label))
    assigned[rows] = True
    prediction[rows] = label
    selected_candidate[rows] = cid
    correct = int(np.sum(y[rows] == label))
    trace.append({**rule, "assigned_examples": int(len(rows)),
                  "incremental_true_positives": correct,
                  "incremental_false_positives": int(len(rows)-correct),
                  "remaining_rows": int((~assigned).sum())})
remaining = np.flatnonzero(~assigned)
prediction[remaining] = top1_by_id[fallback_id][remaining]
selected_candidate[remaining] = fallback_id
trace.append({"order": "fallback", "candidate_id": fallback_id, "label": "checkpoint top-1",
              "assigned_examples": int(len(remaining)),
              "incremental_true_positives": int(np.sum(y[remaining] == prediction[remaining])),
              "incremental_false_positives": int(np.sum(y[remaining] != prediction[remaining])),
              "remaining_rows": 0})
zod01_prediction, _ = _apply_fixed_zod01(all_top1, y, zod01_rules, fallback_id)
metrics = {"zod02_full_validation": _metrics(y, prediction, classes),
           "zod01_full_validation": _metrics(y, zod01_prediction, classes),
           "corrected_vs_zod01": int(np.sum((prediction == y) & (zod01_prediction != y))),
           "regressed_vs_zod01": int(np.sum((prediction != y) & (zod01_prediction == y))),
           "rules": len(rules), "candidate_count": len(package_candidate_ids),
           "fallback_id": fallback_id, "excluded_candidate_counts": exclusions,
           "selection_caveat": "Rules are ranked and evaluated on the same complete V0 validation labels; this is an in-sample selection score."}
metrics["delta_accuracy"] = metrics["zod02_full_validation"]["accuracy"] - metrics["zod01_full_validation"]["accuracy"]
metrics["delta_balanced_accuracy"] = metrics["zod02_full_validation"]["balanced_accuracy"] - metrics["zod01_full_validation"]["balanced_accuracy"]
predictions = pd.DataFrame({"id":validation.id, "prompt_id":validation.prompt_id, "y_true":y,
    "zod01_prediction":zod01_prediction, "zod02_prediction":prediction,
    "selected_candidate_id":selected_candidate, "corrected_by_zod02":(prediction==y)&(zod01_prediction!=y),
    "regressed_by_zod02":(prediction!=y)&(zod01_prediction==y)})
out = Path("artifacts/analysis/zod02_precision_ranked"); out.mkdir(parents=True, exist_ok=True)
pd.DataFrame(trace).to_csv(out/"rule_application_full_v0.csv", index=False)
predictions.to_parquet(out/"predictions_full_v0.parquet", index=False)
(out/"metrics.json").write_text(json.dumps(metrics, indent=2)+"\n")
result = {"metrics":metrics, "rules":rules, "trace":pd.DataFrame(trace), "predictions":predictions}
print(json.dumps({k:v for k,v in metrics.items() if k not in {"zod02_full_validation","zod01_full_validation"}}, indent=2))

{
  "corrected_vs_zod01": 42,
  "regressed_vs_zod01": 98,
  "rules": 385,
  "candidate_count": 22,
  "fallback_id": "ultrabert380_v0_fromeab1ab35_data40a22ce8_split147ce64e_seed2026_cfg34506fe0_codedb9ddff",
  "excluded_candidate_counts": {
    "training_validation_overlap": 3,
    "checkpoint_unavailable": 2,
    "not_evaluation_eligible": 0
  },
  "selection_caveat": "Rules are ranked and evaluated on the same complete V0 validation labels; this is an in-sample selection score.",
  "delta_accuracy": -0.10370370370370363,
  "delta_balanced_accuracy": -0.10370370370370374
}


In [3]:
comparison = pd.DataFrame([
 {"Cascade":"ZOD01","Accuracy":metrics["zod01_full_validation"]["accuracy"],"Balanced accuracy":metrics["zod01_full_validation"]["balanced_accuracy"],"Correct":f'{metrics["zod01_full_validation"]["correct"]}/540'},
 {"Cascade":"ZOD02","Accuracy":metrics["zod02_full_validation"]["accuracy"],"Balanced accuracy":metrics["zod02_full_validation"]["balanced_accuracy"],"Correct":f'{metrics["zod02_full_validation"]["correct"]}/540'}])
comparison["Delta vs ZOD01"]=[0,metrics["delta_balanced_accuracy"]]
display(comparison.style.format({"Accuracy":"{:.2%}","Balanced accuracy":"{:.2%}","Delta vs ZOD01":"{:+.2%}"}))
print(metrics["selection_caveat"])

,Cascade,Accuracy,Balanced accuracy,Correct,Delta vs ZOD01
0,ZOD01,61.30%,61.30%,331/540,+0.00%
1,ZOD02,50.93%,50.93%,275/540,-10.37%


Rules are ranked and evaluated on the same complete V0 validation labels; this is an in-sample selection score.


In [4]:
display(result["trace"].style.format({"precision":"{:.1%}","smoothed_precision":"{:.1%}"}))

,order,candidate_id,sequence_id,label,precision,smoothed_precision,prediction_count,assigned_examples,incremental_true_positives,incremental_false_positives,remaining_rows
0,1,qwen06_v1.1_data72cbc9ee_splitb7cd72d7_seed2026_cfg4df85576_code667e2d0,N0070,gemini-3.1-pro-preview,100.0%,93.8%,14.000000,14,14,0,526
1,2,ultrabert380_v15_from00dfd18d_data40a22ce8_split147ce64e_seed2026_cfge678dbc9_code667e2d0,N0088,gemini-3.1-pro-preview,96.0%,92.6%,25.000000,11,10,1,515
2,3,qwen06_v1.5_data490fe052_split215fd0d0_seed2026_cfg088db19e_code667e2d0,N0072,gpt-6-astra,95.8%,92.3%,24.000000,24,23,1,491
3,4,ultrabert380_v1_all_data3c4b61a6_split0b0fd08e_seed2026_cfgebf0c70f_code1b9bfc2,N0087,gpt-6-astra,95.5%,91.7%,22.000000,3,2,1,488
4,5,ultrabert380_v1_datada64f007_split147ce64e_seed2026_cfgb5a91dee_code667e2d0,N0086,gpt-6-astra,92.0%,88.9%,25.000000,2,1,1,486
5,6,ultrabert380_v15_from00dfd18d_data40a22ce8_split147ce64e_seed2026_cfge678dbc9_code667e2d0,N0088,gpt-6-astra,92.0%,88.9%,25.000000,1,0,1,485
6,7,ultrabert380_v1_datada64f007_split147ce64e_seed2026_cfgb5a91dee_code667e2d0,N0086,gemini-3.1-pro-preview,91.7%,88.5%,24.000000,2,1,1,483
7,8,granite4_h_micro_v1.1_data72cbc9ee_splitb7cd72d7_seed2026_cfgf29156ed_code667e2d0,N0061,gemini-3.1-pro-preview,91.3%,88.0%,23.000000,4,2,2,479
8,9,qwen06_v1.1_data72cbc9ee_splitb7cd72d7_seed2026_cfgf24b2061_code667e2d0,N0071,gemini-3.1-pro-preview,90.0%,86.4%,20.000000,2,0,2,477
9,10,ministral3_3b_v1.1_data72cbc9ee_splitb7cd72d7_seed2026_cfgcc9e4f28_code667e2d0,N0065,gemini-3.1-pro-preview,89.5%,85.7%,19.000000,1,0,1,476


In [5]:
z1,z2=metrics["zod01_full_validation"]["per_label"],metrics["zod02_full_validation"]["per_label"]
per_label=pd.DataFrame([{"Label":l,"ZOD01 recall":z1[l]["recall"],"ZOD02 recall":z2[l]["recall"],"Correct ZOD01":z1[l]["true_positive"],"Correct ZOD02":z2[l]["true_positive"],"Delta":z2[l]["recall"]-z1[l]["recall"]} for l in z1])
display(per_label.style.format({"ZOD01 recall":"{:.1%}","ZOD02 recall":"{:.1%}","Delta":"{:+.1%}"}))

,Label,ZOD01 recall,ZOD02 recall,Correct ZOD01,Correct ZOD02,Delta
0,claude-fable-5,60.0%,6.7%,18,2,-53.3%
1,claude-opus-4-6,96.7%,100.0%,29,30,+3.3%
2,claude-opus-4-8,50.0%,63.3%,15,19,+13.3%
3,claude-opus-5,56.7%,86.7%,17,26,+30.0%
4,claude-sonnet-5,83.3%,80.0%,25,24,-3.3%
5,gemini-3.1-pro-preview,86.7%,96.7%,26,29,+10.0%
6,gemini-3.6-flash,83.3%,36.7%,25,11,-46.7%
7,gemini-3.7-flash,50.0%,20.0%,15,6,-30.0%
8,gemini-3.8-flash,56.7%,60.0%,17,18,+3.3%
9,glm-5.3,33.3%,0.0%,10,0,-33.3%


In [6]:
changed=result["predictions"].loc[result["predictions"].corrected_by_zod02|result["predictions"].regressed_by_zod02]
print("Corrected:",metrics["corrected_vs_zod01"],"Regressed:",metrics["regressed_vs_zod01"],"Changed rows:",len(changed))
display(changed.sort_values(["corrected_by_zod02","id"],ascending=[False,True]))

Corrected: 42 Regressed: 98 Changed rows: 140


,id,prompt_id,y_true,zod01_prediction,zod02_prediction,selected_candidate_id,corrected_by_zod02,regressed_by_zod02
3,02149f5e4a5a,p018,qwen3.8-max,gemini-3.6-flash,qwen3.8-max,granite4_h_micro_v1.1_data72cbc9ee_splitb7cd72...,True,False
22,0b4d1dd59809,p018,claude-opus-5,claude-fable-5,claude-opus-5,smollm3_3b_v0_data40a22ce8_split147ce64e_seed2...,True,False
43,156e486b986b,p072,claude-opus-4-6,claude-sonnet-5,claude-opus-4-6,qwen06_v1.5_data490fe052_split215fd0d0_seed202...,True,False
48,16d413ad1fb0,p057,claude-sonnet-5,claude-fable-5,claude-sonnet-5,smollm3_3b_v1.5_data490fe052_split215fd0d0_see...,True,False
63,1e4b027a5ff8,p015,glm-5.3-flash,claude-fable-5,glm-5.3-flash,smollm3_3b_v1.5_data490fe052_split215fd0d0_see...,True,False
...,...,...,...,...,...,...,...,...
521,fa8bd2783791,p015,gpt-5.6-sol,gpt-5.6-sol,gpt-6-astra,ultrabert380_v0_fromeab1ab35_data40a22ce8_spli...,False,True
522,fa9c30a28959,p005,gemini-3.7-flash,gemini-3.7-flash,gemini-3.1-pro-preview,qwen06_v1.5_data490fe052_split215fd0d0_seed202...,False,True
530,fca21b899581,p018,glm-5.3-flash,glm-5.3-flash,qwen3.8-max,gemma4_e2b_v1.1_data72cbc9ee_splitb7cd72d7_see...,False,True
537,fe0d803e8e69,p084,gemini-3.8-flash,gemini-3.8-flash,claude-opus-4-6,svc_v1.1_data72cbc9ee_splitb7cd72d7_seed2026_c...,False,True
